# Target trial emulation: systemic estradiol initiation and mortality (All of Us)
Eligible: women aged 45-59 (AGE_MIN/AGE_MAX), >= 2 years of EHR before time zero, no prior
systemic estrogen, no history of breast or endometrial cancer, VTE or stroke.
Treatment: initiate systemic estradiol (oral, transdermal, injectable; vaginal estradiol acetate ring).
Comparator (COMPARATOR):
  'vaginal' = initiators of vaginal (local) estrogen with no systemic estrogen on/before that day
  'nonuser' = risk-set matched non-users (same birth year, eligible and not on systemic estrogen at that date)
Outcomes: all-cause death; incident VTE and breast cancer (positive controls, expected HR > 1),
CHD, stroke, hip fracture (expected HR < 1).
WHI-like benchmark: rerun with AGE_MIN, AGE_MAX = 60, 80.
Python builds the cohort and weights; models run in R (e2_tte_aou_models.R). Check lines marked # CHECK.

In [ ]:
# 1. Setup and parameters
import os
import numpy as np, pandas as pd
from google.cloud import bigquery

BQ_dataset = os.environ['WORKSPACE_CDR']   # original run: CDR R2025Q4R6
client = bigquery.Client(default_query_job_config=bigquery.QueryJobConfig(default_dataset=BQ_dataset))
def q(sql): return client.query(sql).result().to_dataframe()

COMPARATOR  = 'vaginal'      # 'vaginal' or 'nonuser'
CUTOFF      = '2023-10-01'   # CHECK: same as the zoster notebooks
STUDY_START = '2017-01-01'
AGE_MIN, AGE_MAX = 45, 60    # benchmark against WHI: 60, 80
LOOKBACK    = 730            # covariate window; >= 1 visit required
EHR_HISTORY = 730            # washout: >= 2 years of EHR with no systemic estrogen before time zero
N_CONTROLS, OVERSAMPLE, SEED = 5, 3, 2026   # nonuser option only
GRACE_DAYS  = None
TAG = f'{COMPARATOR}_{AGE_MIN}_{AGE_MAX}'

lc = "LOWER(concept_name)"
EST  = f"domain_id = 'Drug' AND {lc} NOT LIKE '%ethinyl%'"
E2   = f"{EST} AND {lc} LIKE '%estradiol%'"
OTH  = (f"{EST} AND ({lc} LIKE '%conjugated estrogens%' OR {lc} LIKE '%esterified estrogens%'"
        f" OR {lc} LIKE '%estropipate%' OR {lc} LIKE '%estriol%')")
VAG  = f"{lc} LIKE '%vaginal%'"
SYS_ROUTE = f"REGEXP_CONTAINS({lc}, r'oral|transdermal|topical|inject|patch|pump|gel|spray|emulsion|vaginal')"
FLU   = f"{lc} LIKE '%influenza%' AND (domain_id = 'Drug' OR (domain_id = 'Procedure' AND {lc} LIKE '%vaccin%'))"
MAMMO = f"domain_id = 'Procedure' AND {lc} LIKE '%mammogra%'"
LABEL = f"""CASE
      WHEN {E2} AND {VAG} AND {lc} NOT LIKE '%estradiol acetate%' THEN 'est_vag'
      WHEN {E2} AND {SYS_ROUTE} THEN 'est_sys'
      WHEN {E2} THEN 'est_unknown'
      WHEN {OTH} AND {VAG} THEN 'oth_vag'
      WHEN {OTH} THEN 'oth_sys'
      WHEN {FLU} THEN 'flu'
      WHEN {MAMMO} THEN 'mammo' END"""
CS = f"SELECT concept_id, concept_name, {LABEL} AS label FROM `concept` WHERE ({E2}) OR ({OTH}) OR ({FLU}) OR ({MAMMO})"

In [ ]:
# 2. Check estrogen concepts by label (inspect: est_sys = systemic estradiol, est_vag = local)
check = q(f"""
WITH cs AS ({CS}),
ev AS (
  SELECT drug_concept_id AS cid, person_id FROM `drug_exposure` WHERE drug_concept_id IN (SELECT concept_id FROM cs)
  UNION ALL SELECT drug_source_concept_id, person_id FROM `drug_exposure` WHERE drug_source_concept_id IN (SELECT concept_id FROM cs))
SELECT cs.label, cs.concept_id, cs.concept_name, COUNT(DISTINCT ev.person_id) AS n_persons
FROM ev JOIN cs ON ev.cid = cs.concept_id
WHERE cs.label LIKE 'est%' OR cs.label LIKE 'oth%'
GROUP BY 1, 2, 3 ORDER BY label, n_persons DESC
""")
pd.set_option('display.max_rows', 300, 'display.max_colwidth', 90)
display(check.groupby('label').head(25))
display(check.groupby('label').n_persons.sum())

In [ ]:
# 3. Build the cohort with covariates and outcome dates (one BigQuery script)
CCI = {
  'mi': (1, r'^(I21|I22|I252)'),
  'chf': (1, r'^(I099|I110|I130|I132|I255|I420|I42[5-9]|I43|I50|P290)'),
  'pvd': (1, r'^(I70|I71|I731|I738|I739|I771|I790|I792|K551|K558|K559|Z958|Z959)'),
  'cevd': (1, r'^(G45|G46|H340|I6)'),
  'dementia': (1, r'^(F0[0-3]|F051|G30|G311)'),
  'copd': (1, r'^(I278|I279|J4[0-7]|J6[0-7]|J684|J701|J703)'),
  'rheum': (1, r'^(M05|M06|M315|M3[2-4]|M351|M353|M360)'),
  'pud': (1, r'^K2[5-8]'),
  'liver_mild': (1, r'^(B18|K70[0-39]|K71[3-57]|K73|K74|K760|K76[2-489])'),
  'dm': (1, r'^E1[0-4][01689]'),
  'dm_comp': (2, r'^E1[0-4][23457]'),
  'hemi': (2, r'^(G041|G114|G80[12]|G81|G82|G83[0-49])'),
  'renal': (2, r'^(I120|I131|N03[2-7]|N05[2-7]|N18|N19|N250|Z49[0-2]|Z940|Z992)'),
  'cancer': (2, r'^(C[01][0-9]|C2[0-6]|C3[0-4]|C3[7-9]|C4[01]|C43|C4[5-9]|C5[0-8]|C6[0-9]|C7[0-6]|C8[1-5]|C88|C9[0-7])'),
  'liver_sev': (3, r'^(I850|I859|I864|I982|K704|K711|K721|K729|K76[5-7])'),
  'mets': (6, r'^(C7[7-9]|C80)'),
  'aids': (6, r'^B2[0-24]'),
}
OUTC = {   # person-level first-ever date; used for exclusions, covariates and outcomes
  'vte': r'^(I26|I80[1-3]|I824|I829)',
  'breast_ca': r'^C50',
  'endom_ca': r'^(C54|C55)',
  'stroke': r'^(I63|I64)',
  'chd': r'^I2[0-5]',
  'hip_fx': r'^S72[0-2]',
  'hyster': r'^Z9071',
  'menop': r'^N95',
  'osteop': r'^M8[01]',
}
def case_sql(m, key=lambda v: v):
    return "\n        ".join(f"WHEN REGEXP_CONTAINS(REPLACE(sc.concept_code, '.', ''), r'{key(v)}') THEN '{k}'"
                             for k, v in m.items())
cci_case  = case_sql(CCI, key=lambda v: v[1])
out_case  = case_sql(OUTC)
cci_pivot = ", ".join(f"MAX(IF(x.cat = '{nm}', 1, 0)) AS {nm}" for nm in CCI)
cci_sel   = ", ".join(f"IFNULL(cc.{nm}, 0) AS {nm}" for nm in CCI)
out_pivot = ", ".join(f"MIN(IF(cat = '{k}', d, NULL)) AS first_{k}" for k in OUTC)
out_sel   = ", ".join(f"o.first_{k}" for k in OUTC)

def elig(a, d):
    return f"""{a}.enrollment_date <= {d}
    AND ({a}.death_date IS NULL OR {a}.death_date > {d})
    AND {a}.first_visit <= DATE_SUB({d}, INTERVAL {EHR_HISTORY} DAY)
    AND {a}.last_visit  >= DATE_SUB({d}, INTERVAL {LOOKBACK} DAY)"""
def in_window_age(a, d):
    return f"""{d} >= DATE '{STUDY_START}' AND {d} < DATE '{CUTOFF}'
    AND DATE_DIFF({d}, {a}.birth, DAY) / 365.25 >= {AGE_MIN}
    AND DATE_DIFF({d}, {a}.birth, DAY) / 365.25 <  {AGE_MAX}"""

TR = f"""tr AS (
  SELECT p.*, p.first_e2sys AS d FROM pers p
  WHERE p.first_e2sys IS NOT NULL AND p.first_anysys = p.first_e2sys     -- new user of systemic estradiol
    AND {in_window_age('p', 'p.first_e2sys')} AND {elig('p', 'p.first_e2sys')})"""
if COMPARATOR == 'vaginal':
    COHORT = f"""
WITH {TR},
cm AS (
  SELECT p.*, p.first_vag AS d FROM pers p
  WHERE p.first_vag IS NOT NULL AND (p.first_anysys IS NULL OR p.first_anysys > p.first_vag)
    AND {in_window_age('p', 'p.first_vag')} AND {elig('p', 'p.first_vag')})
SELECT person_id * 10 + 1 AS set_id, d AS index_date, person_id, 1 AS treated, 0 AS draw FROM tr
UNION ALL
SELECT person_id * 10, d, person_id, 0, 0 FROM cm"""
else:
    COHORT = f"""
WITH {TR},
cand AS (
  SELECT t.person_id AS set_id, t.d AS index_date, c.person_id,
    ROW_NUMBER() OVER (PARTITION BY t.person_id ORDER BY
      FARM_FINGERPRINT(CONCAT(CAST(t.person_id AS STRING), '-', CAST(c.person_id AS STRING), '-{SEED}'))) AS draw
  FROM tr t JOIN pers c ON c.byear = t.byear AND c.person_id != t.person_id
  WHERE (c.first_anysys IS NULL OR c.first_anysys > t.d) AND {elig('c', 't.d')})
SELECT person_id AS set_id, d AS index_date, person_id, 1 AS treated, 0 AS draw FROM tr
UNION ALL
SELECT set_id, index_date, person_id, 0, draw FROM cand WHERE draw <= {N_CONTROLS * OVERSAMPLE}"""

sql = f"""
-- (a) estrogen / flu / mammography events
CREATE TEMP TABLE allev AS
WITH cs AS ({CS}),
ev AS (
  SELECT de.person_id, de.drug_exposure_start_date AS d, cs.label FROM `drug_exposure` de JOIN cs ON de.drug_concept_id = cs.concept_id
  UNION ALL SELECT de.person_id, de.drug_exposure_start_date, cs.label FROM `drug_exposure` de JOIN cs ON de.drug_source_concept_id = cs.concept_id
  UNION ALL SELECT po.person_id, po.procedure_date, cs.label FROM `procedure_occurrence` po JOIN cs ON po.procedure_concept_id = cs.concept_id
  UNION ALL SELECT po.person_id, po.procedure_date, cs.label FROM `procedure_occurrence` po JOIN cs ON po.procedure_source_concept_id = cs.concept_id)
SELECT DISTINCT person_id, d, label FROM ev WHERE d IS NOT NULL AND label IS NOT NULL;

-- (b) women: person-level table
CREATE TEMP TABLE pers AS
WITH enroll AS (
  SELECT o.person_id, MIN(o.observation_date) AS enrollment_date
  FROM `observation` o
  JOIN `concept_ancestor` ca ON o.observation_concept_id = ca.descendant_concept_id
  JOIN `concept` c ON ca.ancestor_concept_id = c.concept_id
  WHERE c.concept_name = 'Consent PII' AND c.concept_class_id = 'Module'
  GROUP BY 1),
dth AS (SELECT person_id, MIN(death_date) AS death_date FROM `aou_death` GROUP BY 1),
vs AS (SELECT person_id, MIN(visit_start_date) AS first_visit, MAX(visit_start_date) AS last_visit
       FROM `visit_occurrence` WHERE visit_start_date <= DATE '{CUTOFF}' GROUP BY 1),
e AS (
  SELECT person_id,
    MIN(IF(label = 'est_sys', d, NULL)) AS first_e2sys,
    MIN(IF(label IN ('est_sys', 'est_unknown', 'oth_sys'), d, NULL)) AS first_anysys,
    MIN(IF(label IN ('est_vag', 'oth_vag'), d, NULL)) AS first_vag
  FROM allev GROUP BY 1)
SELECT p.person_id, DATE(p.birth_datetime) AS birth, EXTRACT(YEAR FROM p.birth_datetime) AS byear,
  en.enrollment_date, dt.death_date, vs.first_visit, vs.last_visit,
  e.first_e2sys, e.first_anysys, e.first_vag
FROM `person` p
JOIN `concept` sx ON p.sex_at_birth_concept_id = sx.concept_id
JOIN enroll en ON en.person_id = p.person_id
JOIN vs ON vs.person_id = p.person_id
LEFT JOIN dth dt ON dt.person_id = p.person_id
LEFT JOIN e ON e.person_id = p.person_id
WHERE sx.concept_name = 'Female';

-- (c) cohort: initiators + comparators
CREATE TEMP TABLE cohort0 AS
{COHORT};

-- (d) covariates at time zero and outcome dates
WITH
nvis AS (
  SELECT c.set_id, c.person_id, COUNT(DISTINCT v.visit_start_date) AS n_visits
  FROM cohort0 c JOIN `visit_occurrence` v ON v.person_id = c.person_id
   AND v.visit_start_date >= DATE_SUB(c.index_date, INTERVAL {LOOKBACK} DAY) AND v.visit_start_date < c.index_date
  GROUP BY 1, 2),
ndrug AS (
  SELECT c.set_id, c.person_id, COUNT(DISTINCT de.drug_concept_id) AS n_drugs
  FROM cohort0 c JOIN `drug_exposure` de ON de.person_id = c.person_id
   AND de.drug_exposure_start_date >= DATE_SUB(c.index_date, INTERVAL {LOOKBACK} DAY) AND de.drug_exposure_start_date < c.index_date
  WHERE de.drug_concept_id != 0
  GROUP BY 1, 2),
mark AS (
  SELECT c.set_id, c.person_id,
    MAX(IF(a.label = 'flu', 1, 0)) AS flu,
    MAX(IF(a.label = 'mammo' AND a.d >= DATE_SUB(c.index_date, INTERVAL {LOOKBACK} DAY), 1, 0)) AS mammo
  FROM cohort0 c JOIN allev a ON a.person_id = c.person_id AND a.d < c.index_date AND a.label IN ('flu', 'mammo')
  GROUP BY 1, 2),
dxc AS (
  SELECT person_id, cat, MIN(d) AS first_d FROM (
    SELECT co.person_id, co.condition_start_date AS d,
      CASE
        {cci_case}
      END AS cat
    FROM `condition_occurrence` co JOIN `concept` sc ON co.condition_source_concept_id = sc.concept_id
    WHERE sc.vocabulary_id = 'ICD10CM')
  WHERE cat IS NOT NULL GROUP BY 1, 2),
cci AS (
  SELECT c.set_id, c.person_id, {cci_pivot}
  FROM cohort0 c JOIN dxc x ON x.person_id = c.person_id AND x.first_d < c.index_date
  GROUP BY 1, 2),
outc AS (
  SELECT person_id, {out_pivot} FROM (
    SELECT co.person_id, co.condition_start_date AS d,
      CASE
        {out_case}
      END AS cat
    FROM `condition_occurrence` co JOIN `concept` sc ON co.condition_source_concept_id = sc.concept_id
    WHERE sc.vocabulary_id = 'ICD10CM' AND co.person_id IN (SELECT person_id FROM cohort0))
  WHERE cat IS NOT NULL GROUP BY 1),
svy AS (
  SELECT o.person_id,
    MAX(IF(o.observation_source_concept_id = 1585711, a.concept_name, NULL)) AS srh,
    MAX(IF(o.observation_source_concept_id = 1585857, a.concept_name, NULL)) AS smoke100,
    MAX(IF(o.observation_source_concept_id = 1585940, a.concept_name, NULL)) AS educ
  FROM `observation` o LEFT JOIN `concept` a ON a.concept_id = o.value_source_concept_id
  WHERE o.observation_source_concept_id IN (1585711, 1585857, 1585940)
  GROUP BY 1),
bmi AS (
  SELECT person_id, ANY_VALUE(value_as_number) AS bmi FROM `measurement`
  WHERE measurement_concept_id = 3038553 AND value_as_number BETWEEN 12 AND 80 GROUP BY 1)
SELECT c.set_id, c.index_date, c.person_id, c.treated, c.draw,
  p.birth, p.death_date, p.last_visit, p.first_anysys, p.first_vag,
  IFNULL(nv.n_visits, 0) AS n_visits, IFNULL(nd.n_drugs, 0) AS n_drugs,
  IFNULL(m.flu, 0) AS flu, IFNULL(m.mammo, 0) AS mammo,
  {cci_sel}, {out_sel}, s.srh, s.smoke100, s.educ, b.bmi
FROM cohort0 c
JOIN pers p ON p.person_id = c.person_id
LEFT JOIN nvis  nv ON nv.set_id = c.set_id AND nv.person_id = c.person_id
LEFT JOIN ndrug nd ON nd.set_id = c.set_id AND nd.person_id = c.person_id
LEFT JOIN mark  m  ON m.set_id  = c.set_id AND m.person_id  = c.person_id
LEFT JOIN cci   cc ON cc.set_id = c.set_id AND cc.person_id = c.person_id
LEFT JOIN outc  o  ON o.person_id = c.person_id
LEFT JOIN svy   s  ON s.person_id = c.person_id
LEFT JOIN bmi   b  ON b.person_id = c.person_id
"""
raw = q(sql)
raw.to_pickle(f'e2_aou_{TAG}_raw.pkl')     # after a restart: raw = pd.read_pickle(...)
print(raw.shape, raw.groupby('treated').size().to_dict())

In [ ]:
# 4. Exclusions, comparators, outcomes, covariates
df = raw.copy()
dcols = ['index_date', 'birth', 'death_date', 'last_visit', 'first_anysys', 'first_vag'] + [f'first_{k}' for k in OUTC]
for c in dcols:
    df[c] = pd.to_datetime(df[c])
CUT = pd.Timestamp(CUTOFF)

n0 = df.groupby('treated').size()
excl = np.zeros(len(df), bool)
for k in ['breast_ca', 'endom_ca', 'vte', 'stroke']:            # contraindications / history
    excl |= (df[f'first_{k}'] <= df.index_date).to_numpy()
df = df[~excl & (df.n_visits >= 1)].copy()
print('Excluded (history or no recent visit):', (n0 - df.groupby('treated').size()).to_dict())

if COMPARATOR == 'nonuser':
    df = df.sort_values(['set_id', 'treated', 'draw'], ascending=[True, False, True])
    df['crank'] = (df.treated == 0).astype(int).groupby(df.set_id).cumsum()
    df = df[(df.treated == 1) | (df.crank <= N_CONTROLS)]
    g = df.groupby('set_id').treated
    df = df[(g.transform('max') == 1) & (g.transform('size') > 1)].copy()

end = pd.Series(CUT, index=df.index)
if GRACE_DAYS is not None:
    end = pd.concat([end, df.last_visit + pd.Timedelta(days=GRACE_DAYS)], axis=1).min(axis=1)
df['death'] = (df.death_date.notna() & (df.death_date <= end)).astype(int)
df['end'] = df.death_date.where(df.death == 1, end)
df['time'] = (df.end - df.index_date).dt.days
df = df[df.time > 0].copy()
# per-protocol: comparators censored when they start systemic estrogen
pp_cens = (df.treated == 0) & df.first_anysys.notna() & (df.first_anysys > df.index_date) & (df.first_anysys < df.end)
df['pp_time'] = np.where(pp_cens, (df.first_anysys - df.index_date).dt.days, df.time)
df['pp_death'] = np.where(pp_cens, 0, df.death)

# secondary outcomes: first diagnosis after time zero; prevalent cases flagged and dropped per outcome in R
SEC = ['vte', 'breast_ca', 'chd', 'stroke', 'hip_fx']
for k in SEC:
    f = df[f'first_{k}']
    df[f'prev_{k}'] = (f.notna() & (f <= df.index_date)).astype(int)
    df[f'e_{k}'] = (f.notna() & (f > df.index_date) & (f <= df.end)).astype(int)
    df[f't_{k}'] = np.where(df[f'e_{k}'] == 1, (f - df.index_date).dt.days, df.time)

df.loc[df.dm_comp == 1, 'dm'] = 0
df.loc[df.mets == 1, 'cancer'] = 0
df.loc[df.liver_sev == 1, 'liver_mild'] = 0
df['cci'] = sum(w * df[nm] for nm, (w, _) in CCI.items())
df['cci_cat'] = np.minimum(df.cci, 3).astype(str)
df['cvd_any'] = df[['mi', 'chf', 'pvd', 'cevd']].max(axis=1)
df['dm_any'] = df[['dm', 'dm_comp']].max(axis=1)
df['cancer_any'] = df[['cancer', 'mets']].max(axis=1)
for k in ['hyster', 'menop', 'osteop', 'chd']:
    df[f'hx_{k}'] = (df[f'first_{k}'] < df.index_date).astype(int)
df['prior_vag'] = (df.first_vag < df.index_date).astype(int)
df['age'] = (df.index_date - df.birth).dt.days / 365.25
df['cal_year'] = df.index_date.dt.year.astype(str)
df['lvis'] = np.log1p(df.n_visits)
df['ldrug'] = np.log1p(df.n_drugs)

def srh_map(x):
    x = str(x).lower()
    for k in ['excellent', 'very good', 'good', 'fair', 'poor']:
        if k in x: return k.replace(' ', '_')
    return 'missing'
def smoke_map(x):
    x = str(x).lower().strip()
    return 'ever' if x.endswith('yes') else 'never' if x.endswith('no') else 'missing'
def educ_map(x):
    x = str(x).lower()
    if 'advanced degree' in x or 'college graduate' in x: return 'college_plus'
    if 'college one to three' in x: return 'some_college'
    if 'twelve' in x or 'ged' in x: return 'high_school'
    if any(k in x for k in ['never attended', 'one through', 'five through', 'nine through']): return 'lt_high_school'
    return 'missing'
df['srh_c'] = df.srh.map(srh_map)
df['smoke_c'] = df.smoke100.map(smoke_map)
df['educ_c'] = df.educ.map(educ_map)
df['bmi_c'] = pd.cut(df.bmi, [0, 25, 30, 200], labels=['lt25', '25_30', 'ge30']).astype(str).replace('nan', 'missing')

print(f'Estradiol initiators: {df.treated.sum()}  Comparators ({COMPARATOR}): {(df.treated == 0).sum()}')
summ = df.groupby('treated').agg(n=('death', 'size'), deaths=('death', 'sum'), py=('time', lambda t: t.sum() / 365.25),
                                 age=('age', 'mean'), cci=('cci', 'mean'), visits=('n_visits', 'mean'),
                                 hyster=('hx_hyster', 'mean'), mammo=('mammo', 'mean'))
for k in SEC:
    summ[f'ev_{k}'] = df[df[f'prev_{k}'] == 0].groupby('treated')[f'e_{k}'].sum()
print(summ.round(2).T)

In [ ]:
# 5. Propensity scores (ATT), balance, export for R
import statsmodels.api as sm, statsmodels.formula.api as smf
PS_BASE = ('treated ~ bs(age, df=3) + C(cal_year) + C(cci_cat) + lvis + ldrug + flu + mammo'
           ' + hx_hyster + hx_menop + hx_osteop + hx_chd + cvd_any + dm_any + cancer_any + copd + renal'
           ' + C(smoke_c) + C(educ_c) + C(bmi_c)')
if COMPARATOR == 'nonuser':
    PS_BASE += ' + prior_vag'      # cannot be used vs vaginal initiators (always 0 there by design)
PS_SRH = PS_BASE + ' + C(srh_c)'

def att_weights(formula):
    ps = smf.glm(formula, data=df, family=sm.families.Binomial()).fit().predict(df)
    w = np.where(df.treated == 1, 1.0, ps / (1 - ps))
    cap = np.quantile(w[df.treated == 0], 0.99)
    return np.where((df.treated == 0) & (w > cap), cap, w)
df['w0'] = att_weights(PS_BASE)
df['w'] = att_weights(PS_SRH)

def smd(x, t, w):
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    return (m1 - m0) / np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2)
bal_df = df.assign(**{f'srh_{k}': (df.srh_c == k).astype(int) for k in ['excellent', 'very_good', 'good', 'fair', 'poor']},
                   ever_smoke=(df.smoke_c == 'ever').astype(int))
bal_vars = ['age', 'n_visits', 'n_drugs', 'cci', 'flu', 'mammo', 'hx_hyster', 'hx_menop', 'hx_osteop', 'hx_chd',
            'prior_vag', 'cvd_any', 'dm_any', 'cancer_any', 'ever_smoke',
            'srh_excellent', 'srh_very_good', 'srh_good', 'srh_fair', 'srh_poor']
t, one = bal_df.treated.values, np.ones(len(bal_df))
balance = pd.DataFrame({v: {'smd_unweighted': smd(bal_df[v].values, t, one),
                            'smd_w_no_srh': smd(bal_df[v].values, t, bal_df.w0.values),
                            'smd_w_srh': smd(bal_df[v].values, t, bal_df.w.values)} for v in bal_vars}).T.round(3)
display(balance)
balance.to_csv(f'e2_aou_{TAG}_balance.csv')

cols = (['set_id', 'person_id', 'treated', 'time', 'death', 'pp_time', 'pp_death', 'w', 'w0',
         'age', 'cci', 'lvis', 'ldrug', 'flu', 'mammo', 'hx_hyster']
        + [f'{p}_{k}' for k in SEC for p in ('t', 'e', 'prev')])
df[cols].to_csv(f'e2_aou_{TAG}_model.csv', index=False)
print('Exported', f'e2_aou_{TAG}_model.csv')